
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 2L: Introduction to Delta Lake

In this lab, you will gain hands-on experience with Delta Lake's key features:

- Creating Delta tables
- Performing basic operations (INSERT, UPDATE, DELETE)
- Executing MERGE operations
- Using time travel capabilities
- Optimizing performance

We'll be using the customer data from the TPC-H dataset to work through these concepts.

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will also set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.
<br></br>

```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

In [0]:
%run  ./Includes/Classroom-Setup-Lab

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Course Catalog:,
Your Schema:,


## B. Creating Delta Tables

In this section, you'll create a Delta table from the TPC-H customers dataset. You'll transform the data to use more intuitive column names and filter for a specific subset of customers.

In [0]:
from pyspark.sql.functions import *

spark.sql("DROP TABLE IF EXISTS delta_customers")

customers_df = (spark.table("samples.tpch.customer")
  .select(
    col("c_custkey").alias("customer_id"),
    col("c_name").alias("name"),
    col("c_address").alias("address"),
    col("c_nationkey").alias("nation_id"),
    col("c_phone").alias("phone"),
    col("c_acctbal").alias("account_balance"),
    col("c_mktsegment").alias("market_segment"),
    col("c_comment").alias("comment")
  )
  .filter(col("c_mktsegment").isin("BUILDING", "HOUSEHOLD"))
  .limit(5000)
)

## Write the DataFrame as a Delta table
customers_df.write.mode("overwrite").saveAsTable("delta_customers")

In [0]:
# Inspect the data in the Delta table
display(spark.table("delta_customers"))

customer_id,name,address,nation_id,phone,account_balance,market_segment,comment
412445,Customer#000412445,"0QAB3OjYnbP6mA0B,kgf",21,31-421-403-4333,5358.33,BUILDING,arefully blithely regular epi
412449,Customer#000412449,"zAt1nZNG01gOhIqgyDtDa S,Y0VSofZJs1dd",14,24-710-983-5536,4973.84,HOUSEHOLD,"refully final theodolites. final, slow excuses sleep quickly! quickly ironic idea"
412450,Customer#000412450,fUD6IoGdtF,20,30-293-696-5047,4406.28,BUILDING,refully final dolphins after the carefully bold packages sleep quickly express deposits. fluffily
412451,Customer#000412451,W2Ge0Qd8adH,20,30-590-724-6711,2290.38,BUILDING,slow asymptotes will are carefully final packages. slyly regular fox
412455,Customer#000412455,sGVkj7CxYpfh 5H,16,26-667-672-4269,5456.87,BUILDING,ts. furiously daring multipliers haggle along the slyly thin
412459,Customer#000412459,4LvWWOu4f5eCKxTVVx,2,12-555-921-6728,1902.71,BUILDING,deposits are fluffily across the blithely s
412460,Customer#000412460,9c472xgetdryhb,21,31-422-845-1602,-219.53,BUILDING,s about the foxes x-ray slyly fluffily special Tiresias. regul
412461,Customer#000412461,ND9dWEoPtVlDLCH,7,17-662-733-9810,4218.85,HOUSEHOLD,"hin packages haggle blithely after the furiously final dependencies. regular, regular packages are quickl"
412467,Customer#000412467,"F9F9B ,csN8G7ocZnc7iEXD",9,19-333-990-5481,1098.31,BUILDING,areful platelets. furiously even a
412477,Customer#000412477,JuwfOSCyqIpRgVkR6Vu,16,26-595-532-2007,7666.53,BUILDING,iously special theodolites. quickly pending dependencies wake quickly bold dolphins. express foxes x-ray furi


## C. Exploring Table Metadata

Let's examine the metadata of our newly created Delta table.


In [0]:
%sql
-- Inspecting the table metadata
DESCRIBE EXTENDED delta_customers

col_name,data_type,comment
customer_id,bigint,null
name,string,null
address,string,null
nation_id,bigint,null
phone,string,null
account_balance,"decimal(18,2)",null
market_segment,string,null
comment,string,null
,,
# Delta Statistics Columns,,


In [0]:
%sql
DESCRIBE DETAIL delta_customers

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,d3554d5e-fc30-481b-bbe9-0f697c5aebee,dbacademy.labuser10806356_1751473610.delta_customers,null,s3://unity-catalogs-us-west-2/metastore/4155592-root/3d799779-9ae0-4952-a8ff-e94676f40e23/tables/4f63451c-988d-4de7-a8ae-4bd960ecaf48,2025-07-02T17:06:45.968Z,2025-07-02T17:06:51Z,List(),List(),1,414600,Map(delta.enableDeletionVectors -> true),3,7,List(deletionVectors),"Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)"


In [0]:
%sql
DESCRIBE HISTORY delta_customers

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2025-07-02T17:06:51Z,75073666343693,labuser10806356_1751473610@vocareum.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(808895421739560),0702-162742-g2nbdijh,null,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 5000, numOutputBytes -> 414600)",null,Databricks-Runtime/15.4.x-scala2.12


## D. Basic Delta Operations

Now you'll learn how to perform basic operations on Delta tables, including an `UPDATE` operation.


In [0]:
%sql
-- Increasing their account_balance by 10%
UPDATE delta_customers
SET account_balance = account_balance * 1.1
WHERE market_segment = 'BUILDING';

-- Deleting customer with a negative account balance
DELETE FROM delta_customers
WHERE account_balance < 0;

num_affected_rows
461


## E. Time Travel Operations

Now let's explore Delta Lake's time travel capabilities to view and query previous versions of our data.


In [0]:
%sql
SELECT * FROM delta_customers VERSION AS OF 1

customer_id,name,address,nation_id,phone,account_balance,market_segment,comment
412449,Customer#000412449,"zAt1nZNG01gOhIqgyDtDa S,Y0VSofZJs1dd",14,24-710-983-5536,4973.84,HOUSEHOLD,"refully final theodolites. final, slow excuses sleep quickly! quickly ironic idea"
412461,Customer#000412461,ND9dWEoPtVlDLCH,7,17-662-733-9810,4218.85,HOUSEHOLD,"hin packages haggle blithely after the furiously final dependencies. regular, regular packages are quickl"
412478,Customer#000412478,0j6926BAkzQRQsOgWe3rY,11,21-496-456-5405,6396.57,HOUSEHOLD,s. slyly regular dinos sleep slyly slyly regular excus
412479,Customer#000412479,"ML8gRT,YH7lkddNdSi",2,12-457-273-6401,2254.30,HOUSEHOLD,ideas. final instructions cajole slyly final platelets-- fl
412482,Customer#000412482,"4HHrKtHBPi,mytPJy1FTZe4Wzc2RkHFMQ",20,30-849-795-3196,4977.67,HOUSEHOLD,the regular accounts. furiously express frays boost carefully carefully special packages: quickl
412484,Customer#000412484,TzIIvU4CntHdP,9,19-301-402-7043,3395.44,HOUSEHOLD,ent requests poach across the q
412485,Customer#000412485,"zftac,5s5e,XAkRzLMVpVA0xq9iv",23,33-566-745-5990,8096.44,HOUSEHOLD,". even, express packages are furiously express requests. carefully bold ideas are blithely. pendi"
412486,Customer#000412486,a0X6suTbSxadnhzSZX Yxsa 4xn,24,34-917-763-7300,4882.18,HOUSEHOLD,. fluffily even accounts solve furiously.
412491,Customer#000412491,"1F00, FpHrt9S7ELKizfz,QfYHV2Hbwd0NrU0m",4,14-512-634-5547,2462.20,HOUSEHOLD,"ending ideas. ironic, ironic instructions x-ray. express, ironic foxes wake quickly final courts. blithely bold depo"
412492,Customer#000412492,8ZoIcmVuTzKIFY5incs5yXMNJ8,19,29-996-341-1943,9096.34,HOUSEHOLD,blithely express theodolites wake along the furiously bold


In [0]:
%sql
SELECT COUNT(*) as record_count FROM delta_customers VERSION AS OF 1;

record_count
5000


## F. Restore Operations

Let's learn how to recover data by restoring to a previous version.


In [0]:
%sql
-- Checking the current count for the table
SELECT COUNT(*) as current_count FROM delta_customers;

current_count
4539


In [0]:
%sql
RESTORE TABLE delta_customers TO VERSION AS OF 1;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
621749,2,1,2,415798,621749


In [0]:
%sql
-- Checking count again for the table
SELECT COUNT(*) as current_count FROM delta_customers;

current_count
5000


## G. Using the MERGE Function

Now let's explore Delta Lake's powerful `MERGE` capabilities for upsert operations.


In [0]:
%sql
-- Setup a temporary view to simulated incoming updated customer data
CREATE OR REPLACE TEMPORARY VIEW updated_customers AS
SELECT 999901 AS customer_id, 
       "Customer#000999901" AS name,
       "123 Updated Street" AS address, -- Updated address
       1 AS nation_id,
       "1-123-456-7890" AS phone,
       CAST(10000.00 AS DECIMAL(18,2)) AS account_balance, -- Updated balance
       "BUILDING" AS market_segment,
       "Updated customer record via MERGE" AS comment -- Updated comment
UNION ALL
-- Add a new record that doesn't exist in the table yet
SELECT 999999 AS customer_id,
       "Customer#000999999" AS name,
       "999 Merge Street" AS address,
       5 AS nation_id,
       "1-999-999-9999" AS phone,
       CAST(15000.00 AS DECIMAL(18,2)) AS account_balance,
       "HOUSEHOLD" AS market_segment,
       "New customer added via MERGE" AS comment;


In [0]:
%sql
---- Perform a MERGE operation to update existing customers address, account_balance and comment fields and insert new customers
MERGE INTO delta_customers AS target
USING updated_customers AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
  UPDATE SET 
    address = source.address,
    account_balance = source.account_balance,
    comment = source.comment
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, address, nation_id, phone, account_balance, market_segment, comment)
  VALUES (source.customer_id, source.name, source.address, source.nation_id, 
          source.phone, source.account_balance, source.market_segment, source.comment);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
-- Verify the results
SELECT * FROM delta_customers WHERE customer_id IN (999901, 999999);

customer_id,name,address,nation_id,phone,account_balance,market_segment,comment
999901,Customer#000999901,123 Updated Street,1,1-123-456-7890,10000.00,BUILDING,Updated customer record via MERGE
999999,Customer#000999999,999 Merge Street,5,1-999-999-9999,15000.00,HOUSEHOLD,New customer added via MERGE



&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
